In [ ]:
import sys, os, glob, shutil
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
name = torch.cuda.get_device_name(0); print("GPU:", name, flush=True)
if "T4" not in name and "L4" not in name and "A100" not in name:
    raise SystemExit(f"нужна T4, выдали {name}")
code = os.path.dirname(glob.glob("/kaggle/input/**/train_ce_large.py", recursive=True)[0])
os.makedirs("/kaggle/working/src", exist_ok=True)
for p in glob.glob(code + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
open("/kaggle/working/src/__init__.py", "a").close()
pack = os.path.dirname(glob.glob("/kaggle/input/**/item_texts.parquet", recursive=True)[0])
os.makedirs("/kaggle/working/pack", exist_ok=True)
for p in glob.glob(pack + "/*"):
    dst = "/kaggle/working/pack/" + os.path.basename(p)
    if not os.path.exists(dst): os.symlink(p, dst)

# Чекпоинт первого этапа лежит плоско с префиксом: раскладываем в каталог, как ждёт
# `from_pretrained`. Обучение стартует со второго этапа — предобучение на LLM-парах уже
# сделано и повторять его четыре часа незачем.
hard = os.path.dirname(glob.glob("/kaggle/input/**/stage1__model.safetensors", recursive=True)[0])
os.makedirs("/kaggle/working/stage1", exist_ok=True)
for f in ("model.safetensors", "config.json", "tokenizer.json", "tokenizer_config.json"):
    dst = f"/kaggle/working/stage1/{f}"
    if not os.path.exists(dst): os.symlink(f"{hard}/stage1__{f}", dst)

os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")
sys.argv = ["train_ce_large", "--prepacked", "/kaggle/working/pack", "--holdout-fold", "0",
            # Токенизатор скрипт берёт из --base-model, а интернета в ядре нет: указываем
            # свой каталог, там лежат и config.json, и tokenizer.json.
            "--base-model", "/kaggle/working/stage1",
            "--resume-from", "/kaggle/working/stage1",
            "--hard-negatives", f"{hard}/hard_neg_strict.parquet",
            "--batch-size", "128", "--max-length", "256",
            "--human-epochs", "2", "--human-learning-rate", "1e-5",
            "--output", "/kaggle/working/ce_hardneg"]
from src.train_ce_large import main
main()
